# Experimental Design and Causal Basics

Notebook for **Chapter 12** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 Correlation, Causation and Potential Outcomes

## 12.2 Confounding in Action: a Simulation

In [ ]:
import numpy as np
rng=np.random.default_rng(12)
N=100000
power=rng.random(N)<0.3
adopt=rng.random(N)<np.where(power,0.7,0.2)
base=np.where(power,0.30,0.05)
conv=rng.random(N)<(base+0.02*adopt)
print("share power",power.mean(),"adopt rate",adopt.mean())
print("naive diff",conv[adopt].mean()-conv[~adopt].mean(), conv[adopt].mean(),conv[~adopt].mean())
adj=0
for s in (True,False):
    m=power==s
    a=conv[m&adopt].mean();b=conv[m&~adopt].mean()
    print("power" if s else "regular", m.sum(), a,b,a-b)
    adj=adj+(a-b)*m.mean()
print("adjusted",adj)

In [ ]:
import numpy as np
rng=np.random.default_rng(12)
N=100000
power=rng.random(N)<0.3
adopt=rng.random(N)<np.where(power,0.7,0.2)
base=np.where(power,0.30,0.05)
conv=rng.random(N)<(base+0.02*adopt)   # same draws as the first block, so the random stream continues
rand=rng.random(N)<0.5
conv2=rng.random(N)<(base+0.02*rand)
print("randomized diff",conv2[rand].mean()-conv2[~rand].mean(),conv2[rand].mean(),conv2[~rand].mean())
print("balance power share rand",power[rand].mean(),power[~rand].mean(),"adopt",power[adopt].mean(),power[~adopt].mean())

randomized diff 0.018161492916233563 0.14341147291783343 0.12524998000159987
balance power share rand 0.3014641171293703 0.29749620030397567 adopt 0.6026772917502723 0.13703658199465552


## 12.3 Designing a Good Experiment

In [ ]:
import numpy as np
from scipy.stats import norm
p1,p2=0.10,0.12
za,zb=norm.ppf(0.975),norm.ppf(0.80)
pb=(p1+p2)/2
n=((za*np.sqrt(2*pb*(1-pb))+zb*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2)/(p2-p1)**2
print("n per arm",n)

n per arm 3840.847482436276


## 12.4 Difference-in-Differences

In [ ]:
import numpy as np
rng=np.random.default_rng(12)
n=500
reg=np.repeat([0,1],2*n);post=np.tile(np.repeat([0,1],n),2)
mu=np.array([[400,370],[420,330]])          # rows: control, treated; columns: before, after
y=mu[reg,post]+rng.normal(0,40,len(reg))
X=np.c_[np.ones(len(y)),reg,post,reg*post]
b=np.linalg.lstsq(X,y,rcond=None)[0]
res=y-X@b; se=np.sqrt(res.var(ddof=4)*np.diag(np.linalg.inv(X.T@X)))
print("coef",b)
print("se",se,"ci",b[3]-1.96*se[3],b[3]+1.96*se[3])

## 12.5 Other Causal Tools, and What You Do Not Need Yet

Randomization is the gold standard. When it is impossible, know the names:

## 12.6 The Applied A/B Testing Toolkit

Section 12.3 gave you the basics: pick a metric, size the test, check the split. Real product teams hit five more problems within a month of running experiments. The metric is too noisy to detect anything, someone peeks at the dashboard on day three, the split is slightly off, the metric is a ratio such as revenue per session, or the effect fades after a week. This section gives you a tool for each.

In [ ]:
import numpy as np
rng = np.random.default_rng(2024)
n = 5000                       # users per arm
def arm(effect):
    x = rng.gamma(4, 5, n)                     # pre-experiment spend (mean 20)
    y = 0.6 * x + rng.normal(0, 9, n) + effect  # experiment-period spend
    return x, y
xc, yc = arm(0.0)
xt, yt = arm(1.0)              # true lift: +1.0 per user
# naive difference in means
diff = yt.mean() - yc.mean()
se = np.sqrt(yt.var(ddof=1)/n + yc.var(ddof=1)/n)
# CUPED: theta from pooled data, then adjust each user
x = np.concatenate([xc, xt]); y = np.concatenate([yc, yt])
theta = np.cov(x, y)[0, 1] / x.var(ddof=1)
adj = lambda xx, yy: yy - theta * (xx - x.mean())
ac, at = adj(xc, yc), adj(xt, yt)
diff2 = at.mean() - ac.mean()
se2 = np.sqrt(at.var(ddof=1)/n + ac.var(ddof=1)/n)
rho = np.corrcoef(x, y)[0, 1]
print("correlation rho  :", round(rho, 3))
print("theta            :", round(theta, 3))
print("naive lift, SE   :", round(diff, 3), round(se, 3))
print("CUPED lift, SE   :", round(diff2, 3), round(se2, 3))
print("variance ratio   :", round((se2/se)**2, 3), " 1 - rho^2 =", round(1 - rho**2, 3))

correlation rho  : 0.555
theta            : 0.606
naive lift, SE   : 0.94 0.214
CUPED lift, SE   : 0.977 0.178
variance ratio   : 0.691  1 - rho^2 = 0.692


In [ ]:
import numpy as np
from scipy.stats import chisquare
# Overall split looks mildly off; is it the whole experiment, or one segment?
# rows: segment, users in control, users in treatment (planned 50/50)
data = {"desktop": (30210, 30190), "iOS": (14980, 15070), "Android": (9810, 8250)}
tot_c = sum(v[0] for v in data.values()); tot_t = sum(v[1] for v in data.values())
print("overall", tot_c, tot_t, "p =", round(chisquare([tot_c, tot_t]).pvalue, 6))
for k, (c, t) in data.items():
    stat, p = chisquare([c, t])
    print(f"{k:8s} {c:6d} {t:6d} ratio {t/c:.3f} chi2 {stat:7.2f} p {p:.6f}")

overall 55000 53510 p = 6e-06
desktop   30210  30190 ratio 0.999 chi2    0.01 p 0.935141
iOS       14980  15070 ratio 1.006 chi2    0.27 p 0.603633
Android    9810   8250 ratio 0.841 chi2  134.75 p 0.000000


In [ ]:
import numpy as np
from scipy.optimize import brentq
rng = np.random.default_rng(7)
K, reps = 5, 200000
# A/A test: standardized z statistic at K equally spaced looks.
# Partial sums of N(0,1) blocks: z_k = S_k / sqrt(k)
S = np.cumsum(rng.standard_normal((reps, K)), axis=1)
z = S / np.sqrt(np.arange(1, K + 1))
naive = (np.abs(z) > 1.96).any(axis=1).mean()
pocock_c = brentq(lambda c: (np.abs(z) > c).any(axis=1).mean() - 0.05, 2.0, 3.0)
obf_c = brentq(lambda c: (np.abs(z) > c * np.sqrt(K / np.arange(1, K + 1))).any(axis=1).mean() - 0.05, 1.5, 3.0)
print("peek 5 times at z>1.96  : false positive rate", round(naive, 3))
print("Pocock constant boundary:", round(pocock_c, 3))
print("O'Brien-Fleming c       :", round(obf_c, 3))
print("OBF boundaries per look :", np.round(obf_c * np.sqrt(K / np.arange(1, K + 1)), 2))
print("false positive, Pocock  :", round((np.abs(z) > pocock_c).any(axis=1).mean(), 3))
print("false positive, OBF     :", round((np.abs(z) > obf_c*np.sqrt(K/np.arange(1,K+1))).any(axis=1).mean(), 3))

peek 5 times at z>1.96  : false positive rate 0.142
Pocock constant boundary: 2.413
O'Brien-Fleming c       : 2.033
OBF boundaries per look : [4.55 3.21 2.62 2.27 2.03]
false positive, Pocock  : 0.05
false positive, OBF     : 0.05


In [ ]:
import numpy as np
rng = np.random.default_rng(11)
# mSPRT for a mean difference with known variance sigma^2 = 1, prior N(0, tau^2) on the effect.
tau2, sigma2, alpha = 0.25, 1.0, 0.05
def mix_lr(n, xbar):
    return np.sqrt(sigma2 / (sigma2 + n * tau2)) * np.exp(n**2 * tau2 * xbar**2 / (2 * sigma2 * (sigma2 + n * tau2)))
N, reps = 3000, 2000
def rejects(true_effect):
    x = rng.normal(true_effect, 1, (reps, N))
    n = np.arange(1, N + 1)
    xbar = np.cumsum(x, axis=1) / n
    lr = mix_lr(n, xbar)
    return (lr >= 1 / alpha).any(axis=1).mean(), (lr >= 1/alpha).argmax(axis=1)[(lr >= 1/alpha).any(axis=1)]
fp, _ = rejects(0.0)
pw, stop = rejects(0.1)
print("effect 0.0, monitored at every user: false positive rate", round(fp, 3))
print("effect 0.1: power by n=3000", round(pw, 3), " median stopping n", int(np.median(stop)) + 1)

effect 0.0, monitored at every user: false positive rate 0.035
effect 0.1: power by n=3000 0.988  median stopping n 874


In [ ]:
import numpy as np
rng = np.random.default_rng(5)
n = 4000                                     # users in one arm
sessions = rng.poisson(3, n) + 1             # x_i: sessions per user
taste = rng.normal(0, 1, n)                  # users differ, so one user's sessions are alike
per_session = [rng.normal(2 + 0.8 * u, 1.5, s).clip(0) for s, u in zip(sessions, taste)]
revenue = np.array([p.sum() for p in per_session])
R = revenue.sum() / sessions.sum()           # revenue per session
# delta method: Var(R) ~ (var_y + R^2 var_x - 2 R cov_xy) / (n * mean_x^2)
mx = sessions.mean()
vy, vx = revenue.var(ddof=1), sessions.var(ddof=1)
cxy = np.cov(revenue, sessions)[0, 1]
se_delta = np.sqrt((vy + R**2 * vx - 2 * R * cxy) / (n * mx**2))
# bootstrap users: resample users, keep each user's (revenue, sessions) pair together
boots = []
for _ in range(2000):
    i = rng.integers(0, n, n)
    boots.append(revenue[i].sum() / sessions[i].sum())
se_boot = np.std(boots, ddof=1)
# WRONG: treat every session as an independent row
all_sessions = np.concatenate(per_session)
se_wrong = all_sessions.std(ddof=1) / np.sqrt(len(all_sessions))
print("revenue per session   :", round(R, 4))
print("SE delta method       :", round(se_delta, 4))
print("SE bootstrap (users)  :", round(se_boot, 4))
print("SE wrong (sessions iid):", round(se_wrong, 4))

revenue per session   : 2.1181
SE delta method       : 0.0163
SE bootstrap (users)  : 0.0163
SE wrong (sessions iid): 0.0121


## 12.7 Causal Inference from Observational Data: End to End

Sometimes you cannot randomize: the feature already shipped, the policy is national, ethics forbid it. Then you need methods that rely on assumptions. In this section every example is a simulation, so we know the true effect and can see which method recovers it. Keep one idea in mind: **each method buys credibility by stating one assumption you can argue about**.

In [ ]:
import numpy as np
rng = np.random.default_rng(3)
n = 50000
talent = rng.normal(0, 1, n)
looks = rng.normal(0, 1, n)                      # independent of talent by construction
famous = (talent + looks + rng.normal(0, 0.5, n)) > 1.5   # collider: caused by both
print("corr(talent, looks), everyone      :", round(np.corrcoef(talent, looks)[0, 1], 3))
print("corr(talent, looks), famous only   :", round(np.corrcoef(talent[famous], looks[famous])[0, 1], 3))
print("share famous                       :", round(famous.mean(), 3))

corr(talent, looks), everyone      : -0.011
corr(talent, looks), famous only   : -0.566
share famous                       : 0.158


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(42)
n = 20000
age = rng.normal(35, 8, n)
tenure = rng.exponential(12, n)
activity = rng.normal(0, 1, n)
# who gets the premium feature depends on covariates (confounding)
logit = -1.0 + 0.9 * activity + 0.04 * (tenure - 12) + 0.02 * (age - 35)
t = rng.random(n) < 1 / (1 + np.exp(-logit))
# outcome: spend. True causal effect of the feature = +2.0
y = 20 + 4 * activity + 0.3 * tenure + 0.1 * age + 2.0 * t + rng.normal(0, 5, n)
X = np.column_stack([age, tenure, activity])
Z = (X - X.mean(0)) / X.std(0)
def smd(a, b, wa=None, wb=None):
    wa = np.ones(len(a)) if wa is None else wa; wb = np.ones(len(b)) if wb is None else wb
    ma, mb = np.average(a, weights=wa), np.average(b, weights=wb)
    va, vb = np.average((a-ma)**2, weights=wa), np.average((b-mb)**2, weights=wb)
    return (ma - mb) / np.sqrt((va + vb) / 2)
print("naive difference      :", round(y[t].mean() - y[~t].mean(), 3))
# regression adjustment
A = np.column_stack([np.ones(n), t, X]); b = np.linalg.lstsq(A, y, rcond=None)[0]
print("regression adjusted   :", round(b[1], 3))
# propensity score
ps = LogisticRegression(C=1e6, max_iter=1000).fit(Z, t).predict_proba(Z)[:, 1]
print("propensity range      :", round(ps.min(), 3), round(ps.max(), 3))
# inverse probability weighting (stabilised)
w = np.where(t, t.mean() / ps, (1 - t.mean()) / (1 - ps))
ipw = np.average(y[t], weights=w[t]) - np.average(y[~t], weights=w[~t])
print("IPW                   :", round(ipw, 3))
# balance table: standardized mean difference before and after weighting
print("covariate   SMD raw  SMD weighted")
for j, name in enumerate(["age", "tenure", "activity"]):
    raw = smd(X[t, j], X[~t, j]); wt = smd(X[t, j], X[~t, j], w[t], w[~t])
    print(f"{name:10s} {raw:7.3f} {wt:9.3f}")
# 1-nearest-neighbour matching on the propensity score (treated matched to controls, with replacement)
lp = np.log(ps / (1 - ps))
ct = np.where(~t)[0]; tr = np.where(t)[0]
order = np.argsort(lp[ct]); sorted_lp = lp[ct][order]
pos = np.clip(np.searchsorted(sorted_lp, lp[tr]), 1, len(ct) - 1)
left, right = sorted_lp[pos - 1], sorted_lp[pos]
pick = np.where(np.abs(lp[tr] - left) <= np.abs(lp[tr] - right), pos - 1, pos)
matched = ct[order][pick]
print("matching ATT          :", round((y[tr] - y[matched]).mean(), 3))

naive difference      : 6.786
regression adjusted   : 2.034
propensity range      : 0.008 0.992
IPW                   : 2.071
covariate   SMD raw  SMD weighted
age          0.127    -0.016
tenure       0.446    -0.005
activity     0.801     0.002
matching ATT          : 2.089


In [ ]:
import numpy as np
rng = np.random.default_rng(8)
n = 6000
score = rng.uniform(0, 100, n)                   # running variable; cutoff at 50 gets a coaching programme
treated = score >= 50
y = 30 + 0.4 * score + 5.0 * treated + rng.normal(0, 6, n)   # true jump = 5
print("naive treated - untreated:", round(y[treated].mean() - y[~treated].mean(), 2))
def local_linear(h):
    m = np.abs(score - 50) <= h
    x = score[m] - 50
    X = np.column_stack([np.ones(m.sum()), treated[m], x, x * treated[m]])
    b = np.linalg.lstsq(X, y[m], rcond=None)[0]
    res = y[m] - X @ b
    se = np.sqrt(res.var(ddof=4) * np.linalg.inv(X.T @ X)[1, 1])
    return b[1], se, m.sum()
for h in (5, 10, 20, 50):
    b, se, k = local_linear(h)
    print(f"bandwidth {h:2d}: jump {b:6.2f}  SE {se:4.2f}  n {k}")

naive treated - untreated: 24.23
bandwidth  5: jump   5.07  SE 0.92  n 667
bandwidth 10: jump   4.49  SE 0.67  n 1275
bandwidth 20: jump   4.40  SE 0.46  n 2489
bandwidth 50: jump   4.64  SE 0.30  n 6000


In [ ]:
import numpy as np
rng = np.random.default_rng(21)
n = 40000
u = rng.normal(0, 1, n)                          # unobserved motivation
z = rng.random(n) < 0.5                          # random nudge (email) = instrument
take = (0.8 * z + 0.9 * u + rng.normal(0, 1, n)) > 0.7   # who actually uses the tool
y = 10 + 3.0 * take + 2.0 * u + rng.normal(0, 2, n)      # true effect of using the tool: +3
x = take.astype(float)
ols = np.polyfit(x, y, 1)[0]
# Wald / IV estimate = effect of nudge on outcome / effect of nudge on take-up
itt = y[z].mean() - y[~z].mean()
first = x[z].mean() - x[~z].mean()
print("OLS (confounded)  :", round(ols, 3))
print("nudge -> outcome  :", round(itt, 3))
print("nudge -> take-up  :", round(first, 3))
print("IV (Wald) estimate:", round(itt / first, 3))
# two-stage least squares done by hand, with the correct standard error
Z = np.column_stack([np.ones(n), z]); xhat = Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
X2 = np.column_stack([np.ones(n), xhat]); b = np.linalg.lstsq(X2, y, rcond=None)[0]
Xr = np.column_stack([np.ones(n), x]); res = y - Xr @ b     # residuals use the REAL x
se = np.sqrt(res.var(ddof=2) * np.linalg.inv(X2.T @ X2)[1, 1])
print("2SLS estimate, SE :", round(b[1], 3), round(se, 3))

OLS (confounded)  : 5.076
nudge -> outcome  : 0.641
nudge -> take-up  : 0.22
IV (Wald) estimate: 2.915
2SLS estimate, SE : 2.915 0.128


In [ ]:
import numpy as np
from scipy.optimize import minimize
rng = np.random.default_rng(13)
T0, T1, J = 24, 12, 8                        # months before / after launch, donor regions
f = np.cumsum(rng.normal(0, 1, T0 + T1)) + 0.3 * np.arange(T0 + T1)   # a shared trend
load = rng.uniform(0.5, 1.5, J + 1)
Y = load[:, None] * f + rng.normal(0, 0.3, (J + 1, T0 + T1)) + 10 * rng.uniform(0.5, 1.5, (J + 1, 1))
Y[0] = 0.6 * Y[3] + 0.4 * Y[6] + rng.normal(0, 0.3, T0 + T1)   # region 0 resembles a mix of donors
Y[0, T0:] -= 4.0                             # region 0 launches a change at month 24: true effect -4
treated, donors = Y[0], Y[1:]
def loss(w): return np.sum((treated[:T0] - w @ donors[:, :T0]) ** 2)
cons = ({"type": "eq", "fun": lambda w: w.sum() - 1},)
res = minimize(loss, np.ones(J) / J, bounds=[(0, 1)] * J, constraints=cons, method="SLSQP")
w = res.x
synth = w @ donors
gap = treated - synth
print("weights:", np.round(w, 2))
print("pre-launch RMSE of fit :", round(np.sqrt(np.mean(gap[:T0] ** 2)), 2))
print("mean post gap (effect) :", round(gap[T0:].mean(), 2))
# placebo test: pretend each donor was treated, compute its post/pre RMSE ratio
def ratio(i):
    others = np.delete(Y, i, axis=0); tr = Y[i]
    r = minimize(lambda w: np.sum((tr[:T0] - w @ others[:, :T0]) ** 2), np.ones(J) / J, bounds=[(0, 1)] * J,
                 constraints=cons, method="SLSQP")
    g = tr - r.x @ others
    return np.sqrt(np.mean(g[T0:] ** 2)) / np.sqrt(np.mean(g[:T0] ** 2))
ratios = np.array([ratio(i) for i in range(J + 1)])
print("post/pre RMSE ratios (region 0 first):", np.round(ratios, 1))
print("rank of region 0:", int((ratios >= ratios[0]).sum()), "of", J + 1)

weights: [0.3  0.   0.57 0.07 0.   0.   0.   0.05]
pre-launch RMSE of fit : 0.28
mean post gap (effect) : -4.13
post/pre RMSE ratios (region 0 first): [14.8  4.1  1.3 10.7  1.2  2.9  1.5  2.2  1. ]
rank of region 0: 1 of 9


## Chapter recap

## Practice set

Problem 12.P.1 Treatment group average 5.5, control 3.5, true ATE 1.5. What is the selection bias?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch12.html